In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp
import obs as obs

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
# using mid

In [ ]:
# config
DATASET = rc.DATASET        # '1461ka'
TRAIN_START = 'mid'         # key of rc.STARTS: 'early', 'mid', 'late', 'balanced' (1461-579 ka, no test set) or 'default' (80:20 split)
N_FEATURES = rc.NFEATURES

PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28, 
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545, 
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06, 
          'dwt levels': 2} # taken from optimize_model.ipynb, the 'mid' start optimization set

STUDY_DIR = rc.RUNS_DIR / '_MCTrials'

# only one config needed here since only seed, training data, and test data change; latter two are bespoke for this notebook
cfg = rc.make_config(
    DATASET,
    seed=42,
    params=PARAMS,   # hyperparameter dict; e.g. an Optuna study's best_params to swap
    train_start=TRAIN_START,
)
paramsa = cfg['params']
pathsa = rc.run_paths(cfg['run_tag'])

print("run tag :", cfg['run_tag'])
print("weights :", pathsa['weights'])
print("params  :", paramsa)

gpr_X = obs.gp_reg('dO18_LR04')
gpr_Y = obs.gp_reg('co2_Yam')
gpr_Z = obs.gp_reg('MgCa_Eld')

In [ ]:
import io, contextlib

def build_emulator(params, cfg):
    """Build and compile an untrained RolloutEmulator (also used to reload saved weights without retraining)."""
    W = cfg['window_size']
    brain = LPNE.CNN_model(W, n_features=N_FEATURES, params=params)
    model = AREmulator.RolloutEmulator(
        brain,
        rollout_length=cfg['rollout_length'],
        vali_rollout_length=cfg['rollout_length_validation'],
        window_size=W,
        n_features=N_FEATURES,
        dwt_gamma=params['dwt gamma'],
        dwt_beta=params['dwt beta'],
        dwt_eps=params['dwt eps'],
        dwt_levels=params['dwt levels'],
        noise_level=cfg['noise_level'],
    )
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=params['learning rate'], clipnorm=1.0))
    _ = model(tf.zeros((1, W, N_FEATURES)))
    return model

def fit_emulator(params, data, cfg, acf_threshold, patience, weights_path, seed=None, quiet=False):
    """
    Build, compile and train one RolloutEmulator on data (from LPNE.prepare_data, or the same dict layout).
    params - model hyperparameters
    data   - training data
    cfg    - config array
    
    seed   - None keeps the current random state (what the Optuna objective does, so every trial starts
             differently); an integer clears the Keras session and resets every seed first, so the same
             seed always gives the same run
    quiet  - suppress the per-epoch progress bars and checkpoint messages

    Returns (model, history, phys_check). model holds the weights of the LAST epoch; the best physically-valid
    epoch is saved to weights_path, which only exists if at least one epoch passed the ACF gate.
    """
    if seed is not None:
        tf.keras.backend.clear_session()
        LPNE.set_seeds(seed)

    model = build_emulator(params, cfg)

    # a weights file left over from an earlier run must never pass as this run's best epoch
    Path(weights_path).unlink(missing_ok=True)
    phys_check = sc.PhysicalAttractorCheckpoint(filepath=str(weights_path), acf_threshold=acf_threshold,
                                                patience=patience)

    with contextlib.redirect_stdout(io.StringIO()) if quiet else contextlib.nullcontext():
        history = model.fit(
            *data['train'],
            validation_data=data['valid'],
            epochs=cfg['epochs'],
            callbacks=[phys_check],
            batch_size=cfg['batch_size'],
            validation_batch_size=cfg['batch_size'],
            shuffle=True,
            verbose=0 if quiet else 'auto',
        )
    return model, history, phys_check

In [ ]:
# generate n = 10 seeds and m = 10 ICs
seeds = np.arange(0, 10, 1)
Xs = np.arange(0, cfg['max_age']+1, 1).reshape(-1,1)
m = 100
# using default config seed to generate sample trajectories
ys_X = gpr_X.sample_y(Xs, m, cfg['seed'])[::-1] # flip samples to ensure oldest -> youngest
ys_Y = gpr_Y.sample_y(Xs, m, cfg['seed'])[::-1]
ys_Z = gpr_Z.sample_y(Xs, m, cfg['seed'])[::-1]


In [ ]:
# the m sampled trajectories of each variable over its GP mean (red dotted), in physical units
ages = Xs[::-1, 0]   # oldest -> youngest, matching the flipped samples
STEP = 10            # plot every STEP-th sample only, for readability
panels = [   # (GP, samples, label)
    (gpr_X, ys_X, f"X: {rc.STATE_RECORDS[0]}"),
    (gpr_Y, ys_Y, f"Y: {rc.STATE_RECORDS[1]}"),
    (gpr_Z, ys_Z, f"Z: {rc.STATE_RECORDS[2]}"),
]

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(3, 1, figsize=(22, 12), sharex=True)

    for ax, (gpr, ys, label), in zip(axs, panels):
        ax.plot(ages, ys[:, ::STEP], color='black', lw=0.8, alpha=0.2)
        ax.plot([], [], color='black', lw=2, alpha=0.2, label=f"GP samples (every {STEP}th: {ys[:, ::STEP].shape[1]} of {ys.shape[1]})")   # one legend entry for all samples
        ax.plot(ages, gpr.predict(Xs)[::-1], color='red', linestyle=':', lw=2.5, label="GP mean")
        ax.set_ylabel(label)
        ax.legend(loc='upper left', fontsize=14)
        ax.grid(True, alpha=0.3)

    axs[-1].invert_xaxis()
    axs[-1].set_xlabel("time [kyr]")
    plt.tight_layout()
    plt.show()

### Setup A - Multiple Training Trajectories
- generate $l$ trajectories from GP 
- use train $m$ models with $n$ seeds s.t. $l=m\times n$
- rollout each model on the same IC (GP mean trajectory)

In [ ]:
# training: every seed trains one model on each of the first M_A sampled trajectories -> n x M_A models.
# Weights go to one file per (seed, trajectory) instead of a list of models (many Keras models won't fit
# comfortably in memory); the testing cell loads them back one at a time
SEED_DIR_A = STUDY_DIR / 'setupA'
SEED_DIR_A.mkdir(parents=True, exist_ok=True)
M_A = 10            # trajectories per seed (the first M_A of the m samples); n x M_A models in total
RETRAIN_A = False   # False: skip (seed, trajectory) pairs whose weights already exist, so an interrupted run resumes

base_A = rc.load_data(cfg)   # GP mean record, oldest -> youngest; supplies the ages, insolation and variances
assert np.array_equal(base_A[rc.AGE_COL].to_numpy(), Xs[::-1, 0]), "GP samples and the record must cover the same ages"

def trajectory_frame(j):
    """the GP mean record with X, Y, Z swapped for sampled trajectory j (insolation and ages unchanged)"""
    df = base_A.copy()
    df[rc.STATE_RECORDS] = np.column_stack([ys_X[:, j], ys_Y[:, j], ys_Z[:, j]])
    return df

def weights_path_A(s, j):
    return SEED_DIR_A / f'seed{s}_traj{j}.weights.h5'

scalers_A = []        # scalers_A[j]: fit on trajectory j's training split (the same for every seed)
histories_A = {}      # (seed, trajectory) -> keras history dict
best_epochs_A = {}    # (seed, trajectory) -> best physically-valid epoch, or None if none passed the ACF gate
for j in tqdm(range(M_A), desc="trajectories"):
    splits_j = rc.split_data(trajectory_frame(j), cfg)
    scaler_j, train_j, valid_j, _ = rc.fit_and_transform_splits(splits_j)
    scalers_A.append(scaler_j)
    X_tr, y_tr, I_tr = LPNE.create_windows(train_j, cfg['window_size'], cfg['rollout_length'])
    X_va, y_va, I_va = LPNE.create_windows(valid_j, cfg['window_size'], cfg['rollout_length_validation'])
    lib = {'train': ([X_tr, I_tr], y_tr), 'valid': ([X_va, I_va], y_va)}

    for s in seeds:
        wpath = weights_path_A(s, j)
        if wpath.exists() and not RETRAIN_A:
            continue
        model, h, check = fit_emulator(cfg['params'], lib, cfg, cfg['acf_threshold'], 15, wpath,
                                       seed=int(s), quiet=True)
        histories_A[(s, j)] = h.history
        if wpath.exists():
            best_epochs_A[(s, j)] = check.best_epoch
        else:
            print(f"seed {s}, trajectory {j}: no epoch passed the ACF gate - saving last-epoch weights")
            best_epochs_A[(s, j)] = None
            model.save_weights(str(wpath))
        del model, h, check

In [ ]:
# testing: each model rolled out from the ICs of the trajectory it was trained on -> rollouts_A[seed, trajectory].
# Each model has its own scaler, so every rollout is mapped back to physical units and then onto the GP mean
# record's scaler (the one Setup B uses), so all of them share one axis
ref_splits = rc.split_data(base_A, cfg)
ref_scaler = rc.fit_scaler(ref_splits)
truth_A = ref_splits['full']        # GP mean record (unscaled); insolation is column 3
fullt_A = ref_splits['time']        # ages [ka], oldest -> youngest
W = cfg['window_size']

emu_A = build_emulator(cfg['params'], cfg)   # one model, reloaded with each (seed, trajectory)'s weights
rollouts_A = np.empty((len(seeds), M_A, truth_A.shape[0], N_FEATURES))
with tqdm(total=len(seeds) * M_A, desc="rollouts") as bar:
    for a, s in enumerate(seeds):
        for j in range(M_A):
            emu_A.load_weights(str(weights_path_A(s, j)))
            ic = scalers_A[j].transform(rc.split_data(trajectory_frame(j), cfg)['full'])
            with contextlib.redirect_stderr(io.StringIO()):   # hide sp.rollout's own per-rollout progress bar
                ro = sp.rollout(emu_A, ic, W)
            rollouts_A[a, j] = ref_scaler.transform(scalers_A[j].inverse_transform(ro))
            bar.update()

In [ ]:
# all n x M_A rollouts, colored by seed (each seed's M_A models form one cluster), with each seed's ensemble
# mean drawn bold on top; GP mean ground truth black dashed. In the GP mean record's scaled units
truth_A_scaled = ref_scaler.transform(truth_A)
TEST_END_A = (ref_splits['test_time'].min() + ref_splits['train_time'].max()) / 2   # test | train boundary [ka]
IC_END_A = (fullt_A[W - 1] + fullt_A[W]) / 2   # end of the seed window: rows before this are each trajectory's ICs
SEED_COLORS = plt.cm.tab10(np.arange(len(seeds)) % 10)

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(4, 1, figsize=(22, 18), sharex=True)

    for k in range(3):
        for a, s in enumerate(seeds):
            axs[k].plot(fullt_A, rollouts_A[a, :, :, k].T, color=SEED_COLORS[a], lw=0.6, alpha=0.12)
            axs[k].plot(fullt_A, rollouts_A[a, :, :, k].mean(axis=0), color=SEED_COLORS[a], lw=2,
                        label=f"seed {s}")
        axs[k].plot(fullt_A, truth_A_scaled[:, k], color='black', linestyle='dashed', lw=2, alpha=0.8,
                    label="GP mean (ground truth)")
        axs[k].set_ylabel(sp.MAPPINGS[k])
        axs[k].grid(True, alpha=0.3)

    # insolation: given, not predicted (the same for every trajectory)
    axs[3].plot(fullt_A, truth_A_scaled[:, 3], color='coral', lw=1.5)
    axs[3].set_title("Insolation forcing")
    axs[3].set_ylabel("Insolation (scaled)")
    axs[3].grid(True, alpha=0.3)

    for ax in axs:
        ax.axvline(TEST_END_A, color='black', linestyle=':', lw=2)
        ax.axvline(IC_END_A, color='dimgray', linestyle=':', lw=2)
    axs[0].annotate('test  |  train / validation', xy=(TEST_END_A, 1.0), xycoords=('data', 'axes fraction'),
                    xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14)
    axs[0].annotate('ICs  |  rollout', xy=(IC_END_A, 1.0), xycoords=('data', 'axes fraction'),
                    xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14, color='dimgray')

    # one legend for the whole figure: thin lines = individual models, bold = that seed's mean over its M_A models
    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(seeds) + 1, fontsize=14, frameon=True,
               title=f"thin: each of a seed's {M_A} models (one per sampled trajectory)   bold: that seed's mean")
    axs[-1].invert_xaxis()
    axs[-1].set_xlabel("time [kyr]")
    plt.tight_layout(rect=[0, 0.07, 1, 1])
    plt.show()

### Setup B - Same Model Tested on Multiple ICs
- use $n$ seeds to train $n$ models on one trajectory (mean GP)
- test each model on $m$ ICs (generated from GP)

In [ ]:
# training
data = LPNE.prepare_data(cfg)          # {'train', 'valid', 'scaler', 'splits'}
SEED_DIR = STUDY_DIR / 'setupB'
SEED_DIR.mkdir(parents=True, exist_ok=True)

models, histories, checks = [], [], []
for s in seeds:
    wpath = SEED_DIR / f'seed{s}.weights.h5'
    model, h, check = fit_emulator(cfg['params'], data, cfg, cfg['acf_threshold'], 15, wpath,
                                   seed=int(s), quiet=True)
    if wpath.exists():
        model.load_weights(str(wpath))   # best physically-valid epoch, not the last one
    else:
        print(f"seed {s}: no epoch passed the ACF gate - keeping last-epoch weights")
    models.append(model); histories.append(h); checks.append(check)

In [ ]:
# testing: every model (seed) rolled out from every sampled IC -> rollouts[i, j] = seed i, IC j
truth = data['splits']['full']    # GP mean record (unscaled), oldest -> youngest; insolation is column 3
insol = truth[:, 3]               # the same forcing drives every rollout
fullt = data['splits']['time']    # ages [ka], oldest -> youngest
assert ys_X.shape[0] == truth.shape[0], "GP samples and the record must cover the same ages"

W = cfg['window_size']
rollouts = np.empty((len(models), m, truth.shape[0], N_FEATURES))
for i, model in enumerate(models):
    for j in range(m):
        ic = np.column_stack([ys_X[:, j], ys_Y[:, j], ys_Z[:, j], insol])   # sample j as an X, Y, Z, insol record
        rollouts[i, j] = sp.rollout(model, data['scaler'].transform(ic), W)

In [ ]:
# all n seeds x m ICs rollouts against the GP mean ground truth, in the training scaler's units
truth_scaled = data['scaler'].transform(truth)
TEST_END = (data['splits']['test_time'].min() + data['splits']['train_time'].max()) / 2   # test | train boundary [ka]
IC_END = (fullt[W - 1] + fullt[W]) / 2   # end of the seed window: rows before this are the sampled ICs [ka]
EMU_COLORS = ["#FFB600", "#6519C5", "#8DB1A7"]   # emulator colors of sp.plot_rollout's palette, for X, Y, Z

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(4, 1, figsize=(22, 15), sharex=True)

    for k in range(3):
        for i in range(rollouts.shape[0]):
            for j in range(rollouts.shape[1]):
                axs[k].plot(fullt, rollouts[i, j, :, k], color=EMU_COLORS[k], lw=0.8, alpha=0.25,
                            label=f"emulator rollouts ({rollouts.shape[0]} seeds x {rollouts.shape[1]} ICs)"
                                  if i == j == 0 else None)
        axs[k].plot(fullt, truth_scaled[:, k], color='black', linestyle='dashed', lw=2, alpha=0.8,
                    label="GP mean (ground truth)")
        axs[k].set_ylabel(sp.MAPPINGS[k])
        axs[k].legend(loc='upper left', fontsize=14)
        axs[k].grid(True, alpha=0.3)

    # insolation: given, not predicted
    axs[3].plot(fullt, truth_scaled[:, 3], color='coral', lw=1.5)
    axs[3].set_title("Insolation forcing")
    axs[3].set_ylabel("Insolation (scaled)")
    axs[3].grid(True, alpha=0.3)

    for ax in axs:
        ax.axvline(TEST_END, color='black', linestyle=':', lw=2)
        ax.axvline(IC_END, color='dimgray', linestyle=':', lw=2)
    axs[0].annotate('test  |  train / validation', xy=(TEST_END, 1.0), xycoords=('data', 'axes fraction'),
                    xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14)
    axs[0].annotate('ICs  |  rollout', xy=(IC_END, 1.0), xycoords=('data', 'axes fraction'),
                    xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14, color='dimgray')

    axs[-1].invert_xaxis()
    axs[-1].set_xlabel("time [kyr]")
    plt.tight_layout()
    plt.show()